# Homework Assignment 4

Heavily based on Technion's CS236781 assignment 3.

Submitted by:

| #       |              Name |             Id |             email |
|---------|-------------------|----------------|------------------ |
|Student 1|  shaked greenstein | 212482632 | greens@post.bgu.ac.il |
|Student 2|  matan idan | 208538611 | idanmat@post.bgu.ac.il |

## Introduction

In this assignment we'll learn to generate text with a deep multilayer RNN network based on GRU cells.
We will then shift to implementing and training a transformer-encoder, which we will use to perform sentiment analysis.

## Datasets

The pre-prepared data required by this assignment (e.g. the `hw4_tests/` used to verify your implementation) is expected to live under a single datasets directory.

This location is configured by the `DATASETS_DIR` constant at the top of `hw4/answers.py`. Its default value is `~/datasets`.

**If your datasets are located elsewhere on the course HPC, edit `DATASETS_DIR` in `hw4/answers.py` so that the check below passes.**

In [3]:
import os
from hw4.answers import DATASETS_DIR

hw4_tests_dir = os.path.join(DATASETS_DIR, 'hw4_tests')
print(f'Looking for: {hw4_tests_dir}')
assert os.path.isdir(hw4_tests_dir), \
    f"'{hw4_tests_dir}' not found - edit DATASETS_DIR in hw4/answers.py to point to your datasets directory."
print('Datasets path OK.')

Looking for: C:\Users\shaked/datasets\hw4_tests


AssertionError: 'C:\Users\shaked/datasets\hw4_tests' not found - edit DATASETS_DIR in hw4/answers.py to point to your datasets directory.

## Libraries
The following commandline will install a few needed libraries in your Python environment. It should install fine, warnings (even red ones) are not to be feared.

In [4]:
! pip install "datasets<5" "pyarrow<23" "transformers<5"

   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   ---------------------------------------- 529.0/529.0 kB 3.3 MB/s  0:00:00
   ---------------------------------------- 0.0/28.0 MB ? eta -:--:--
   ----- ---------------------------------- 3.9/28.0 MB 19.0 MB/s eta 0:00:02
   ----------------- ---------------------- 12.1/28.0 MB 29.0 MB/s eta 0:00:01
   ------------------------------ --------- 21.5/28.0 MB 33.7 MB/s eta 0:00:01
   ---------------------------------------- 28.0/28.0 MB 33.1 MB/s  0:00:00
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------- ----------------- 6.8/12.0 MB 39.0 MB/s eta 0:00:01
   ----------------------------------- ---- 10.7/12.0 MB 24.9 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 23.7 MB/s  0:00:00
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   ---------------------------------------- 566.4/566.4 kB 15.3 MB/s  0:00:00
   ---------------


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Contents
- [Part1: Sequence Models](#part1)
    - [Text generation with a char-level RNN](#part1_1)
    - [Obtaining the corpus](#part1_2)
    - [Data Preprocessing](#part1_3)
    - [Dataset Creation](#part1_4)
    - [Model Implementation](#part1_5)
    - [Generating text by sampling](#part1_6)
    - [Training](#part1_7)
    - [Generating a work of art](#part1_8)
    - [Questions](#part1_9)
- [Part 2: Transformer Encoder](#part2)
    - [Reminder: scaled dot product attention](#part2_1)
    - [Sliding window attention](#part2_2)
    - [Multihead Sliding window attention](#part2_3)
    - [Sentiment analysis](#part2_4)
    - [Obtaining the dataset](#part2_5)
    - [Tokenizer](#part2_6)
    - [Transformer Encoder](#part2_7)
    - [Training](#part2_8)
    - [Questions](#part2_9)


## Submission

When you are **completely done** with the assignment, create your submission file by running the cell below.

Before running it, make sure that you have:
1. Implemented all the code in the `hw4` directory.
2. Run **all** the cells in `Part1_Sequence.ipynb` and `Part2_Transformer.ipynb` from top to bottom, **without exception-throwing errors**. Warnings are ok, but the code must run.
3. Answered all the questions and filled in your name(s) in the cell at the top of this notebook.
4. **Saved every notebook** (including this one) so the latest, fully-executed versions are written to disk.

The cell below will first verify that Part 1 and Part 2 were executed successfully, and only then package all three notebooks (as saved on disk) together with the `hw4` directory into a single zip file for submission. Part 0 does not need to be run — it is included as-is.

In [ ]:
import os, json, zipfile

# ---- Configuration ----
SUBMISSION_FILENAME = 'hw4_submission.zip'
NOTEBOOKS = ['Part0_Intro.ipynb', 'Part1_Sequence.ipynb', 'Part2_Transformer.ipynb']
CODE_DIR = 'hw4'
# Notebooks that must be fully executed before submission (Part0 need not be run).
REQUIRE_EXECUTED = ['Part1_Sequence.ipynb', 'Part2_Transformer.ipynb']
EXCLUDE_DIRS = {'__pycache__', '.ipynb_checkpoints'}


def _execution_problems(nb_path):
    """Return (unexecuted, errored) non-empty code-cell numbers for a notebook on disk."""
    with open(nb_path, encoding='utf-8') as f:
        nb = json.load(f)
    unexecuted, errored, n = [], [], 0
    for cell in nb['cells']:
        if cell.get('cell_type') != 'code' or not ''.join(cell.get('source', [])).strip():
            continue
        n += 1
        if cell.get('execution_count') is None:
            unexecuted.append(n)
        if any(o.get('output_type') == 'error' for o in cell.get('outputs', [])):
            errored.append(n)
    return unexecuted, errored


# ---- Verify Part 1 and Part 2 were executed successfully ----
problems = []
for nb_name in REQUIRE_EXECUTED:
    if not os.path.isfile(nb_name):
        problems.append(f"{nb_name}: file not found."); continue
    unexecuted, errored = _execution_problems(nb_name)
    if unexecuted:
        cells = ", ".join(f"#{c}" for c in unexecuted)
        problems.append(f"{nb_name}: has unexecuted code cells (counting non-empty code cells from the top): {cells}.")
    if errored:
        cells = ", ".join(f"#{c}" for c in errored)
        problems.append(f"{nb_name}: has code cells that raised errors (counting non-empty code cells from the top): {cells}.")

assert not problems, (
    "Submission aborted - your notebooks are not ready:\n"
    + "\n".join('  - ' + p for p in problems)
    + "\n\nRun ALL cells in Part1 and Part2 without errors, SAVE every notebook, then run this cell again."
)

# ---- Build the submission zip ----
with zipfile.ZipFile(SUBMISSION_FILENAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    for nb_name in NOTEBOOKS:
        zf.write(nb_name)
    for root, dirs, files in os.walk(CODE_DIR):
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
        for fn in files:
            zf.write(os.path.join(root, fn))

print(f"Created '{SUBMISSION_FILENAME}' ({os.path.getsize(SUBMISSION_FILENAME) / 1e6:.2f} MB). Contents:")
with zipfile.ZipFile(SUBMISSION_FILENAME) as zf:
    for name in zf.namelist():
        print('  ', name)
